# Unidad 4. Corpus y colecciones documentales extensas

En este cuaderno vas a construir un catálogo de la colección digitalizada de la **Fundación Orellana**, cliente de **Corpus & Contexto**. La colección está organizada en tres subcarpetas (`cartas/`, `manuscritos/`, `prensa/`) con muchos archivos de texto sueltos.

## Qué vas a practicar

1. Recorrer una estructura de carpetas con `pathlib`.
2. Extraer metadatos de cada documento sin leer su contenido en profundidad.
3. Construir un catálogo con Pandas a partir de muchos archivos.
4. Calcular agregados por subcarpeta.
5. Exportar el catálogo documentado.

## 1. Preparación del entorno

In [ ]:
from pathlib import Path

# Si falta pandas, activa el entorno virtual .venv de la asignatura e instala:
# python -m pip install -r 03_unidades/U4_corpus_colecciones_documentales/requirements_U4.txt
import pandas as pd

## 2. Localizar los documentos

In [ ]:
carpeta_coleccion = Path("materiales/coleccion_orellana")

archivos = sorted(carpeta_coleccion.rglob("*.txt"))
len(archivos)

## 3. Extraer metadatos por documento

In [ ]:
registros = []
for archivo in archivos:
    texto = archivo.read_text(encoding="utf-8")
    n_palabras = len(texto.split())
    registros.append({
        "nombre_archivo": archivo.name,
        "subcarpeta": archivo.parent.name,
        "tamano_bytes": archivo.stat().st_size,
        "num_palabras_aprox": n_palabras,
    })

registros[0]

## 4. Construir el catálogo con Pandas

In [ ]:
df_catalogo = pd.DataFrame(registros)
df_catalogo.shape

In [ ]:
df_catalogo.head()

## 5. Agregados por subcarpeta

In [ ]:
resumen = df_catalogo.groupby("subcarpeta").agg(
    num_documentos=("nombre_archivo", "count"),
    palabras_totales=("num_palabras_aprox", "sum"),
).reset_index()

resumen

## 6. Exportar el catálogo

In [ ]:
df_catalogo.to_csv("U4_catalogo_orellana.csv", index=False, encoding="utf-8")
resumen.to_csv("U4_catalogo_orellana_resumen.csv", index=False, encoding="utf-8")

## 7. Diagnóstico

Responde aquí, en pocas frases: ¿qué subcarpeta concentra más documentos? ¿Y más palabras? ¿Coinciden ambas cosas? ¿Qué le dirías a la Fundación Orellana sobre por dónde empezar si, en el futuro, decide encargar un estudio filológico solo de una parte de la colección?

*Escribe tu respuesta aquí.*

## 8. Bloque final opcional: una colección real y grande

Todo lo que has hecho en este cuaderno funciona igual si, en vez de 155 documentos sintéticos, la colección tiene miles de documentos reales. Vamos a comprobarlo con una biblioteca digital real: 1.202 libros en español de dominio público del proyecto Project Gutenberg, disponibles en Hugging Face.

Este bloque es **opcional y no se evalúa**. Para poder ejecutarlo en clase sin que todo el grupo dependa de la misma conexión wifi a la vez, **descarga el dataset antes de la sesión** (una sola vez; después queda guardado en tu ordenador y se carga al instante):

```python
from datasets import load_dataset

load_dataset(
    "parquet",
    data_files={"es": "https://huggingface.co/datasets/manu/project_gutenberg/resolve/main/data/es-00000-of-00001-ad684e007393cf76.parquet"},
    split="es",
)
```

Ejecuta esas líneas desde una terminal, con el entorno `.venv` activado, antes de venir a clase. El archivo pesa unos 190 MB comprimidos.

### Cargar la colección real

In [ ]:
from datasets import load_dataset

gutenberg = load_dataset(
    "parquet",
    data_files={"es": "https://huggingface.co/datasets/manu/project_gutenberg/resolve/main/data/es-00000-of-00001-ad684e007393cf76.parquet"},
    split="es",
)
df_gutenberg = gutenberg.to_pandas()
df_gutenberg.shape

### Extraer metadatos, igual que con la colección de Orellana

In [ ]:
df_gutenberg["num_palabras_aprox"] = df_gutenberg["text"].str.split().str.len()
df_gutenberg[["id", "num_palabras_aprox"]].head()

### Comparar Orellana (sintética) con Gutenberg (real)

In [ ]:
comparacion = pd.DataFrame({
    "colección": ["Orellana (sintética)", "Gutenberg ES (real)"],
    "documentos": [len(df_catalogo), len(df_gutenberg)],
    "palabras_totales": [resumen["palabras_totales"].sum(), df_gutenberg["num_palabras_aprox"].sum()],
})
comparacion

### Un problema real que la colección sintética no tenía

La colección de Orellana está construida a propósito sin duplicados. Una colección real no viene así. Comprueba si `id` tiene valores repetidos:

In [ ]:
df_gutenberg["id"].duplicated().sum()

### Reflexión final (bloque opcional)

Responde en pocas frases: ¿ha cambiado el método que has usado en este cuaderno por trabajar con una colección miles de veces más grande? Si Fundación Orellana te dijera que su colección va a crecer hasta este tamaño, ¿qué comprobarías primero, a la vista de lo que acabas de ver aquí?

*Escribe tu respuesta aquí.*